# Week 5 · Day 3 — Linear Regression by Hand, One Epoch at a Time

Yesterday we learned *what* a model is. Today we build one — the simplest useful model there is — and we do **everything manually**, with no machine-learning library. No `scikit-learn`, no hidden magic. Just NumPy and arithmetic you can follow number by number.

The goal is to *watch* the model learn. We will:
1. Start with a bad line (a random guess).
2. Run **one single step** of learning, by hand — no loops.
3. Look at the new slope and intercept, and draw the new line on the data.
4. Run the **next** step, and the next, and watch the line crawl toward the points until it fits.

By the end you will have seen, with your own eyes, the exact thing that happens inside every machine-learning model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## 1. The data

We keep the dataset tiny — just 5 points — so you can check every calculation by hand if you want to. It's **study hours vs exam marks**: the more hours a student studies, the higher their marks.

- `x` = hours studied
- `y` = marks obtained

In [ ]:
x = np.array([1, 2, 3, 4, 5], dtype=float)      # hours studied
y = np.array([3, 5, 6.5, 9, 10.5])              # marks obtained

for xi, yi in zip(x, y):
    print(f"  studied {xi:.0f} hour(s)  ->  scored {yi}")

In [ ]:
# Always look at the data first
plt.scatter(x, y, color="red", s=80, zorder=3)
plt.xlabel("hours studied (x)")
plt.ylabel("marks (y)")
plt.title("Our 5 data points")
plt.grid(True, alpha=0.3)
plt.show()

The points clearly rise from left to right — a straight line should fit them well. Our job is to **find that line**.

---
## 2. The model: a straight line

A straight line is described by two numbers:

$$ \hat{y} = m \cdot x + b $$

- **m** is the *slope* — how steep the line is (how many extra marks per extra hour of study).
- **b** is the *intercept* — where the line crosses the y-axis (marks for zero hours).
- **ŷ** ("y-hat") is the line's *prediction* for a given x.

This is just `y = mx + c` from school maths. Finding the best line means **finding the best `m` and `b`.**

In [ ]:
def predict(m, b, x):
    """The model: given slope m and intercept b, predict y for each x."""
    return m * x + b

---
## 3. Measuring how wrong a line is: the cost

For any line, each point has an **error** — the gap between the real mark `y` and the line's prediction `ŷ`. To score a whole line with one number, we:

1. take each error `(ŷ − y)`,
2. **square** it (so over- and under-estimates both count, and big misses count more),
3. **average** the squared errors.

That average is the **cost**. A good line has low cost; a bad line has high cost. *(This is the same "deviations → squared → averaged" idea as variance from Week 4.)*

$$ \text{cost} = \frac{1}{2m}\sum (\hat{y} - y)^2 $$

The extra `½` is a convenience that makes the slope maths tidier later — it doesn't change which line is best.

In [ ]:
def cost(m, b, x, y):
    y_hat = predict(m, b, x)
    errors = y_hat - y
    return np.mean(errors ** 2) / 2

---
## 4. How the line improves: the update rule

Here is the one piece of new machinery. To improve a line, we need to know **which way to nudge `m` and `b`** to lower the cost. That direction is given by two quantities (the *gradients*):

$$ \text{grad}_m = \frac{1}{m}\sum (\hat{y} - y)\cdot x \qquad \text{grad}_b = \frac{1}{m}\sum (\hat{y} - y) $$

Then we take a small step in the downhill direction:

$$ m \leftarrow m - \alpha \cdot \text{grad}_m \qquad b \leftarrow b - \alpha \cdot \text{grad}_b $$

- **α** (the *learning rate*) is the size of the step. Small α = slow but safe.
- One pass of this update over all the data is called **one epoch**.

You don't need to derive these formulas today — just understand: **the gradient points uphill, so we step the opposite way to go downhill toward lower cost.**

In [ ]:
def gradients(m, b, x, y):
    y_hat = predict(m, b, x)
    errors = y_hat - y
    grad_m = np.mean(errors * x)
    grad_b = np.mean(errors)
    return grad_m, grad_b

We'll also make a small helper that draws the current line on top of the data, so we can *see* each epoch's result.

In [ ]:
def show_line(m, b, x, y, title=""):
    plt.scatter(x, y, color="red", s=80, zorder=3, label="real data")
    x_line = np.linspace(0, 6, 100)
    plt.plot(x_line, predict(m, b, x_line), color="blue", linewidth=2, label=f"line: y = {m:.3f}x + {b:.3f}")
    plt.xlabel("hours studied"); 
    plt.ylabel("marks")
    plt.title(title) 
    plt.legend() 
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 13)
    plt.show()

---
## 5. The starting point — a deliberately bad guess

Every model has to start *somewhere*. We start with `m = 0` and `b = 0` — a completely flat line lying on the x-axis. It fits the data terribly, and that's the point: we'll watch it improve.

We also fix the learning rate `α = 0.02` (small, so the improvement is gradual and easy to watch).

In [ ]:
m = 0.0      # slope, starting guess
b = 0.0      # intercept, starting guess
alpha = 0.02 # learning rate (step size)

print(f"Starting line:  y = {m:.3f} x + {b:.3f}")
print(f"Starting cost:  {cost(m, b, x, y):.3f}")
show_line(m, b, x, y, title="Epoch 0 — the starting line (before any learning)")

A flat line through the origin — clearly wrong, and the cost is high. Now let's improve it, **one epoch at a time, by hand.**

---
## 6. Epoch 1 — the first manual step

We do this the long way, printing every intermediate number, so you can see exactly what happens. No loop — just the four moves: **predict → measure error → compute gradients → update.**

In [ ]:
# --- EPOCH 1 ---
# Step A: what does the current line predict?
y_hat = predict(m, b, x)
print("predictions :", y_hat)
print("actual y    :", y)

# Step B: the gradients (which way is downhill)
grad_m, grad_b = gradients(m, b, x, y)
print(f"\ngrad_m = {grad_m:.4f}   grad_b = {grad_b:.4f}")

# Step C: take one downhill step — update m and b
m = m - alpha * grad_m
b = b - alpha * grad_b
print(f"\nnew m = {m:.4f}   new b = {b:.4f}")
print(f"new cost = {cost(m, b, x, y):.4f}   (was 26.750)")

In [ ]:
show_line(m, b, x, y, title="After Epoch 1")

The line has **tilted upward** — the slope jumped from 0 to about 0.48. The cost dropped from 26.75 to about 15.6. One step, and it's already less wrong. Let's do the next one.

---
## 7. Epoch 2 — the exact same four moves again

Notice the code below is *identical* to Epoch 1. That repetition is the whole idea — learning is just doing the same small step over and over.

In [ ]:
# --- EPOCH 2 ---
grad_m, grad_b = gradients(m, b, x, y)
print(f"grad_m = {grad_m:.4f}   grad_b = {grad_b:.4f}")

m = m - alpha * grad_m
b = b - alpha * grad_b
print(f"new m = {m:.4f}   new b = {b:.4f}")
print(f"new cost = {cost(m, b, x, y):.4f}")

In [ ]:
show_line(m, b, x, y, title="After Epoch 2")

Steeper still (slope ≈ 0.85), cost down to about 9.1. The line is climbing toward the points.

---
## 8. Epoch 3

In [ ]:
# --- EPOCH 3 ---
grad_m, grad_b = gradients(m, b, x, y)
m = m - alpha * grad_m
b = b - alpha * grad_b
print(f"m = {m:.4f}   b = {b:.4f}   cost = {cost(m, b, x, y):.4f}")

In [ ]:
show_line(m, b, x, y, title="After Epoch 3")

---
## 9. Epochs 4 and 5

By now you know the four moves by heart. Let's run two more, one cell each.

In [ ]:
# --- EPOCH 4 ---
grad_m, grad_b = gradients(m, b, x, y)
m -= alpha * grad_m
b -= alpha * grad_b
print(f"m = {m:.4f}   b = {b:.4f}   cost = {cost(m, b, x, y):.4f}")
show_line(m, b, x, y, title="After Epoch 4")

In [ ]:
# --- EPOCH 5 ---
grad_m, grad_b = gradients(m, b, x, y)
m -= alpha * grad_m
b -= alpha * grad_b
print(f"m = {m:.4f}   b = {b:.4f}   cost = {cost(m, b, x, y):.4f}")
show_line(m, b, x, y, title="After Epoch 5")

Look how the line is now genuinely passing *through* the cloud of points. The slope is near 1.5 and still rising toward its target. Each step is getting smaller — because as we near the bottom of the cost valley, the gradient shrinks, so the steps automatically shorten. The model is slowing down as it homes in.

---
## 10. Spotting the pattern — now we can use a loop

We've now done the same four moves five times by hand. Doing this 50 or 500 times by hand would be silly — this is exactly what a **loop** is for. The loop below does nothing new; it just repeats the identical step automatically, and records the history so we can plot it.

In [ ]:
# Restart from scratch to run the whole thing cleanly
m, b = 0.0, 0.0
alpha = 0.02
history = []   # remember (epoch, m, b, cost) at each step

for epoch in range(1, 51):
    grad_m, grad_b = gradients(m, b, x, y)
    m -= alpha * grad_m
    b -= alpha * grad_b
    history.append((epoch, m, b, cost(m, b, x, y)))

    # print a progress line at a few checkpoints
    if epoch in (1, 5, 10, 20, 30, 50):
        print(f"epoch {epoch:2d}:  m = {m:.3f}   b = {b:.3f}   cost = {cost(m, b, x, y):.4f}")

In [ ]:
show_line(m, b, x, y, title="After 50 epochs — the fitted line")

That is a genuinely good fit — the line runs right through the data. And every step of getting there was the same four moves you did by hand.

---
## 11. Watching the whole journey at once

Two pictures tell the story of the training.

First, **the cost falling** epoch by epoch — this curve is the single best picture of "a model learning." It drops steeply at first (when the line is very wrong) and flattens as the line gets good.

In [ ]:
epochs = [h[0] for h in history]
costs   = [h[3] for h in history]

plt.plot(epochs, costs, color="purple", linewidth=2)
plt.xlabel("epoch"); plt.ylabel("cost")
plt.title("The cost falls as the model learns")
plt.grid(True, alpha=0.3)
plt.show()

Second, **all the lines together** — every line from every epoch, faint at the start (bad fits) and bold at the end (good fit). You can watch the line sweep upward into place.

In [ ]:
plt.scatter(x, y, color="red", s=80, zorder=3, label="data")
x_line = np.linspace(0, 6, 100)

for (epoch, mm, bb, cc) in history:
    shade = epoch / len(history)          # 0 = early, 1 = late
    plt.plot(x_line, mm * x_line + bb,
             color="blue", alpha=0.15 + 0.85 * shade, linewidth=1)

plt.xlabel("hours studied"); plt.ylabel("marks")
plt.title("Every epoch's line — faint (early) to bold (final)")
plt.ylim(0, 13); plt.grid(True, alpha=0.3)
plt.show()

---
## 12. Using the trained model

We now have a trained line — final `m` and `b`. A model is only useful if it can **predict**. Let's ask it: *what mark would a student who studies 3.5 hours get?*

In [ ]:
hours = 3.5
predicted = predict(m, b, hours)
print(f"Final model:  y = {m:.3f} x + {b:.3f}")
print(f"A student who studies {hours} hours is predicted to score {predicted:.1f} marks")

And in plain words, the slope has a real meaning: **each extra hour of study is worth about `m` extra marks**, according to what the model learned from the data.

In [ ]:
print(f"The model learned that each extra hour of study is worth about {m:.2f} marks.")

---
## Summary — what you just did

You built and trained a linear-regression model **completely by hand**, with no machine-learning library:

- The **model** is a straight line, `ŷ = m·x + b`.
- The **cost** measures how wrong a line is: square the errors and average them.
- **One epoch** = four moves: predict → measure error → compute gradients → nudge `m` and `b` downhill.
- Repeating that step (first by hand, then in a loop) makes the cost fall and the line settle onto the data.
- The **learning rate** `α` controls the step size.

Every machine-learning model — no matter how large — is doing a version of exactly this: define a cost, then take small downhill steps to make it small. You've now seen the engine from the inside.

Next time, we'll let a real library (`scikit-learn`) do these steps for us in three lines — but you'll know precisely what it's doing under the hood.